In [1]:
# This script is to generate a helper file which help to transform the Solid data into sample pairs
import pandas as pd
import csv
import os
import glob

In [6]:
# create a csv file with columns name as video_path,start,end,label.  Adjust the path if needed
# This is to calculate the arousal value, generate the label for each time windows, and store the info about each sample pairs. 
# This file will help prepare the dataset in later parts
game_name='fps'
game_file_name = f'{game_name}_pairs.csv'
with open(game_file_name, 'w', newline='') as file:
    writer = csv.writer(file)

    # Write the column names, 'start' is the start time, 'end' is the end time
    writer.writerow(['video_path', 'start', 'end', 'label'])






In [7]:

# Get the path to the folder where the videos are stored, adjust it if needed
folder_path = f'./{game_name}/videos'

# Find all files in the folder
files = glob.glob(os.path.join(folder_path, '*'))

# Get the names of the files
file_names = [os.path.basename(file) for file in files]



In [4]:
gameplay_info = pd.read_csv('clean_data.csv')

/tmp/ipykernel_1234207/2248557627.py:1: DtypeWarning: Columns (23,24,98,99,100) have mixed types. Specify dtype option on import or set low_memory=False.
  gameplay_info = pd.read_csv('clean_data.csv')


In [8]:
# This is to generate entries about 3-second time windows in Solid from the clean_data.csv
down = 0
up = 0
total = 0
for file in file_names:
  file_path = str(folder_path)+'/'+str(file)
  print("processing:",file)
  file_name = file[0:-5]
  # change game name
  id_list = file_name.split(f'_{game_name}_')

  # Find the entries with Solid game sessions
  df1 = gameplay_info.loc[(gameplay_info['[control]player_id']==id_list[0])&(gameplay_info['[control]session_id']==id_list[1])]
  
# Four data per 1 second, interval step is to 4, i represents data index, we want to get the first index in each second
  for i in range(0,len(df1),4):
    # If there is no overlap, the first data index is i at the beginning of every 6 seconds(one sample pair) and the last data index is i+23;
    # i+24 is then the index of the first data of the next 6 seconds. 
    if ((i+23)<len(df1)):
      # Here, To calculate start time and end time of each time window
      start = float(i)
      start = start/4.0
      end = float(i+24)
      end = end/4.0

      # Calculate the arousal value of the corresponding time window based on the time and index.
      df2 = df1.iloc[i:i+24]
      arousal_1 = 0
      arousal_2 = 0

      # The index range for the first 3 seconds is 0-11, including 11, for a total of 12 entries of data
      for j in range(0,12):
        df3 = df2.iloc[j]
        arousal = df3['[output]arousal']
        arousal_1 = arousal_1+arousal
      # average
      arousal_1 = arousal_1/12
      #The index range for the last 3 seconds is 12-23
      for k in range(12,24):
        df3 = df2.iloc[k]
        arousal = df3['[output]arousal']
        arousal_2 = arousal_2+arousal
      arousal_2 = arousal_2/12
      # keep labels simple, arousal unchanging-->"same", arousal decreasing-->"down", arousal increasing-->"up"
      arousal_change = "same"
      if(arousal_1>arousal_2):
        arousal_change = "down"
        down = down+1
      if(arousal_1<arousal_2):
        arousal_change = "up"
        up = up+1
      total = total + 1

      with open(game_file_name, 'a', newline='') as file:
        writer = csv.writer(file)
        writer.writerow([file_path, start, end, arousal_change])

  # As two pairs cannot extract corresponding frames from videos, then removed in later process
  # print("down:",down,"; up:",up, "total:", total)









processing: 77F6B31E-B228-3C6D-A66F-E7E6FDED3587_fps_577EF511-FB38-2E40-EBFB-44EB6212FE03.webm
processing: FFAB0F54-EFF2-FA1A-1892-F9D8ED3C9B31_fps_84DBFACE-808A-1FEA-0CC0-E6AE75D27B7A.webm
processing: E49C1269-8AB7-8C8F-FC88-AA92C2F1E260_fps_4C5AC2A2-4D5D-F68B-2DBC-5C15425F406C.webm
processing: B06410CC-D778-33B2-B1FD-A828EE4AF413_fps_17B2C42C-7404-90F6-6589-F26BF580A4CC.webm
processing: CD5833CD-93D3-318C-371A-18A32DEFC38B_fps_8E6A4014-86C8-1398-8F96-17B97739CA40.webm
processing: BD35F47E-4F73-8664-B165-98D2BF2CDA48_fps_E49C5044-42E8-0EDD-A6D6-9EC3B7D25814.webm
processing: 72B642AD-2D4A-574F-05EC-EC93C0C52335_fps_48E5F72F-2B57-AC6C-C4D6-1203CC831164.webm
processing: C5A540EA-A98F-9BE7-0AF4-CEC3DA884958_fps_7F96E814-2B1B-C50E-53FB-FACCF451CE83.webm
processing: 5F222604-CB86-1ABA-EB7B-31FF78B7BE93_fps_C697A0AC-C316-A3C7-1E20-9767B3E5600D.webm
processing: 718078D1-70DB-2916-366D-1119A22DCD7F_fps_A4C63961-86C6-C302-5A36-533C5BB8C944.webm
processing: 99827717-1B21-7ECC-CAD1-99489F7E19EA_f